# MedVis Exercise Sheet 2 — Task 5: Morphological Operations

Goal: Apply **Erosion**, **Dilation**, **Opening**, and **Closing** to medical PNG images using two different kernel shapes — Rectangular and Elliptical.

## Preparation

In [ ]:
!pip install opencv-python

import cv2 as cv
import matplotlib.pyplot as plt

## Define Images, Kernels and Operations

We store all images, kernels, and operations in dictionaries so we can loop through them cleanly without repeating code.

In [ ]:
# Images from dataset2 — make sure these are uploaded to your Colab session
images = {
    'Lumbar Bones':  'dataset2/4_lumbar_bones.png',
    'Blood Vessels': 'dataset2/brain_blood_vessels.png',
    'Brain Tumor':   'dataset2/brain_tumor.png',
    'Lung':          'dataset2/lung.png',
}

# Two kernel shapes as required by the task
kernels = {
    'Rectangular (5x5)': cv.getStructuringElement(cv.MORPH_RECT,    (5, 5)),
    'Elliptical (5x5)':  cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5)),
}

# Four morphological operations
operations = {
    'Erosion':  lambda img, k: cv.erode(img, k, iterations=1),
    'Dilation': lambda img, k: cv.dilate(img, k, iterations=1),
    'Opening':  lambda img, k: cv.morphologyEx(img, cv.MORPH_OPEN,  k),
    'Closing':  lambda img, k: cv.morphologyEx(img, cv.MORPH_CLOSE, k),
}

## Apply Morphological Operations to All Images

For each image we create a figure with 2 rows (one per kernel) and 5 columns (original + 4 operations).

In [ ]:
for img_name, img_path in images.items():

    # Load image as grayscale
    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)

    # Create subplot grid: 2 rows (kernels) x 5 columns (original + 4 operations)
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle(img_name, fontsize=16, fontweight='bold')

    for row, (kernel_name, kernel) in enumerate(kernels.items()):

        # Column 0: original image
        axes[row, 0].imshow(img, cmap='gray')
        axes[row, 0].set_title(f'Original\n({kernel_name})', fontsize=9)
        axes[row, 0].axis('off')

        # Columns 1-4: apply each operation
        for col, (op_name, op_fn) in enumerate(operations.items(), start=1):
            result = op_fn(img, kernel)
            axes[row, col].imshow(result, cmap='gray')
            axes[row, col].set_title(op_name, fontsize=9)
            axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()

## Discussion

### What does each operation do?

**Erosion** shrinks white regions by removing pixels at their edges. Any white pixel that does not have enough white neighbors (as defined by the kernel) gets turned black. Thin structures narrower than the kernel disappear entirely.

**Dilation** expands white regions by adding pixels at their edges. Any black pixel touching a white region gets turned white. Thin gaps between nearby structures close up.

**Opening** (erosion then dilation) removes small white noise dots and thin connections while keeping large structures roughly the same size. Useful for cleaning up segmentation results.

**Closing** (dilation then erosion) fills small black holes inside white regions while keeping large structures roughly the same size. Useful for filling gaps inside segmented organs.

### Rectangular vs Elliptical kernel

The **rectangular kernel** affects all directions equally including corners, making it the most aggressive. The **elliptical kernel** has a rounded shape so it treats diagonal directions more gently, producing smoother results that better match the natural curved shapes of anatomical structures.

### Observations per image

**Lumbar Bones** — Erosion widens gaps between vertebrae and breaks thin connections. Dilation merges vertebrae together. Opening removes small bone fragments. Closing fills small gaps inside bone regions.

**Blood Vessels** — Erosion removes thin vessels, only thick ones survive. Dilation thickens vessels and merges nearby ones. Opening simplifies the vascular network by removing the thinnest branches. Closing bridges small gaps between close vessels.

**Brain Tumor** — Erosion shrinks the white brain region and removes small isolated patches. Dilation expands the white region and fills small dark holes. Closing is most useful here — fills internal dark gaps cleanly.

**Lung** — Closing shows the most dramatic effect — the small black holes (airways) inside the lung regions get filled, producing clean solid lung shapes. Opening removes small white speckles scattered around.